In [3]:
%pip install --upgrade jupyter-server

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install fastapi uvicorn pydantic google-cloud-bigquery google-genai

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install --upgrade pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from google.cloud import bigquery
from google import genai
from google.genai import types
from dotenv import load_dotenv

In [7]:
# Instantiate the central FastAPI application
app = FastAPI(
    title="Retail AI Agent Backend",
    description="Secure BigQuery and Gemini orchestration service for retail business analysis"
)

In [23]:
# Attach Cross-Origin Resource Sharing (CORS) middleware to the FastAPI application
app.add_middleware(
    CORSMiddleware,
    # allow_origins=["*"] permits requests from local browsers (e.g., file:// or localhost) and hosted web pages
    allow_origins=["*"],
    # allow_credentials=True enables session headers/cookies if future authentication is added
    allow_credentials=True,
    # allow_methods=["*"] allows all standard HTTP methods including GET, POST, and OPTIONS preflights
    allow_methods=["*"],
    # allow_headers=["*"] permits common request headers like Content-Type and Authorization
    allow_headers=["*"],
)


In [28]:
import os

# Rename the files cleanly to standard dotfiles
if os.path.exists(".env.txt"):
    os.rename(".env.txt", ".env")
    print("Successfully renamed: .env.txt -> .env")

if os.path.exists(".gitignore.txt"):
    os.rename(".gitignore.txt", ".gitignore")
    print("Successfully renamed: .gitignore.txt -> .gitignore")

Successfully renamed: .env.txt -> .env
Successfully renamed: .gitignore.txt -> .gitignore


In [29]:
# Loads variables from your local .env into system memory
load_dotenv()

# Client automatically finds os.environ["GEMINI_API_KEY"]
ai_client = genai.Client()

In [30]:
# ------------------------------------------------------------------------------
# Environment Variables, Cloud Clients & System Prompts
# ------------------------------------------------------------------------------

# Fetch the Google Cloud Project ID from the environment, with a safe fallback placeholder
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "your-gcp-project-id")

# Specify the target BigQuery dataset containing the 13 tables and analytical views
DATASET_ID = os.getenv("BQ_DATASET_ID", "retail_dataset")

# Initialize the BigQuery client using Application Default Credentials (ADC)
bq_client = bigquery.Client(project=PROJECT_ID)

# Initialize the Gemini GenAI client (picks up GEMINI_API_KEY from environment automatically)
ai_client = genai.Client()

/opt/homebrew/anaconda3/lib/python3.10/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [34]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [42]:
import pypdf
from google.genai import types


# 1. Extract text from your PDF
pdf_path = "Retail AI Agent - Context File for Agent.pdf"  # Replace with your PDF file name/path

reader = pypdf.PdfReader(pdf_path)
system_instruction_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        system_instruction_text += text + "\n"

print(f"Extracted {len(system_instruction_text)} characters from system instruction PDF.")

my_question = "What is the day with highest revenue?"

# 2. Use in Gemini generate_content call
response = ai_client.models.generate_content(
    model="gemini-3.6-flash",
    contents= my_question,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction_text,
        temperature=0.1,
    ),
)

print(response.text)

Extracted 28376 characters from system instruction PDF.
To identify the day with the highest revenue, we query the pre-aggregated daily sales table (**`rtl_agg_overall_sales_kpis_daily`**) to ensure maximum accuracy while minimizing BigQuery scan costs (well under the $1 limit).

### Recommended SQL Query

```sql
SELECT 
    trans_dt AS peak_revenue_date,
    total_sales_amt AS highest_gross_revenue,
    total_transactions,
    avg_trans_value
FROM 
    `your_project.your_dataset.rtl_agg_overall_sales_kpis_daily`
ORDER BY 
    total_sales_amt DESC
LIMIT 1;
```

---

### Key Business Context & Potential Drivers to Investigate

Once this date is retrieved, typical underlying drivers for peak sales days in our retail operations include:

1. **Promotional Campaigns:** Was there a major sitewide or channel-specific marketing push (e.g., BOGO, Flash Sale) active on `rtl_promo_table` or `rtl_marketing_data`?
2. **Channel Contribution:** Did the spike come primarily from In-Store physical loca

In [36]:
# ------------------------------------------------------------------------------
# 3. Security Helper & Request Validation Schema
# ------------------------------------------------------------------------------

# Define the Pydantic data model to enforce the incoming POST request structure
class QueryRequest(BaseModel):
    # Enforces that the incoming JSON must contain a non-empty string under the key 'user_message'
    user_message: str


# Programmatic guardrail function to inspect and validate raw SQL before execution
def validate_sql(sql: str) -> bool:
    # Convert query to uppercase and strip leading/trailing whitespace for consistent evaluation
    clean_sql = sql.strip().upper()
    
    # Verify that the query begins with a read-only keyword (SELECT or WITH for CTEs)
    if not (clean_sql.startswith("SELECT") or clean_sql.startswith("WITH")):
        return False  # Reject query if it attempts any command other than reading
        
    # List of forbidden DDL/DML keywords that modify or destroy database structures/data
    blocked_keywords = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE", "MERGE", "CREATE"]
    
    # Split query into individual string tokens while stripping semicolons
    tokens = clean_sql.replace(";", "").split()
    
    # Iterate through blocked keywords and reject the query if any are present
    for kw in blocked_keywords:
        if kw in tokens:
            return False  # Reject execution immediately
            
    # Return True if all read-only validation criteria pass
    return True

In [37]:
# ------------------------------------------------------------------------------
# 4. API Endpoints
# ------------------------------------------------------------------------------

# Define the GET endpoint triggered when the user clicks 'Run Agent Analysis' on the frontend
@app.get("/api/dashboard-kpis")
def get_dashboard_kpis():
    """Fetches high-level metrics for dashboard cards and drill-down charts directly from the summary view."""
    
    # Construct a high-performance aggregation query targeting the pre-computed view
    query = f"""
    SELECT 
        channel,
        SUM(net_sales) AS net_sales,
        SUM(total_orders) AS total_orders,
        ROUND(SUM(net_sales) / NULLIF(SUM(total_orders), 0), 2) AS aov
    FROM `{PROJECT_ID}.{DATASET_ID}.v_daily_executive_kpis`
    GROUP BY channel
    """
    
    # Set a maximum bytes billed threshold (10 MB) to guarantee near-zero query cost
    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=10 * 1024 * 1024)
    
    try:
        # Submit the query job to BigQuery with the bounded cost configuration
        query_job = bq_client.query(query, job_config=job_config)
        
        # Convert BigQuery rows into a Pandas DataFrame
        df = query_job.to_dataframe()
        
        # Convert the DataFrame into a JSON-compatible list of dictionaries and return to frontend
        return df.to_dict(orient="records")
        
    except Exception as e:
        # Return HTTP 500 error if BigQuery fails to process the view
        raise HTTPException(status_code=500, detail=f"BigQuery aggregation failed: {str(e)}")


# Define the POST endpoint for interactive natural language queries from the chat UI
@app.post("/api/agent-chat")
def agent_chat(req: QueryRequest):
    """Translates user natural language into SQL, runs it safely, and synthesizes executive insights."""
    
    # Step A: Text-to-SQL Conversion
    # Prompt the LLM to generate standard BigQuery SQL for the specific question
    sql_prompt = f"Write a BigQuery SQL query to answer the following business question: '{req.user_message}'. Output only SQL."
    
    # Call Gemini 1.5 Flash using a low temperature for strict deterministic SQL syntax
    sql_response = ai_client.models.generate_content(
        model="gemini-1.5-flash",
        contents=sql_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.1  # Low temperature minimizes hallucination and syntax errors
        )
    )
    
    # Strip Markdown code block backticks to extract the clean SQL statement
    raw_sql = sql_response.text.replace("```sql", "").replace("```", "").strip()
    
    # Step B: Programmatic Security Check
    # Verify that the generated SQL complies with read-only rules before touching the database
    if not validate_sql(raw_sql):
        return {
            "error": "The generated query violated database read-only security policies.",
            "sql_used": raw_sql,
            "data_preview": []
        }
    
    # Step C: BigQuery Execution with Hard Cost Limits
    # Limit max scan size to 100 MB per dynamic query to keep costs strictly under $1.00
    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=100 * 1024 * 1024)
    
    try:
        # Execute query against BigQuery
        query_job = bq_client.query(raw_sql, job_config=job_config)
        
        # Convert the query result iterator into a list of row dictionaries
        results = [dict(row) for row in query_job.result()]
        
    except Exception as e:
        # Return helpful error information if the SQL query fails to execute
        return {
            "error": f"Failed executing query on BigQuery: {str(e)}",
            "sql_used": raw_sql,
            "data_preview": []
        }
        
    # Step D: Executive Business Synthesis
    # Prepare a synthesis prompt containing the user question, SQL, and top 20 rows of results
    synth_prompt = f"""
    User Question: {req.user_message}
    SQL Executed: {raw_sql}
    Data Output (first 20 rows): {results[:20]}
    
    Provide:
    1. Executive Key Findings (clear bullet points).
    2. 2-3 Actionable Strategic Recommendations (for merchandising, marketing, or operations).
    """
    
    # Call Gemini 1.5 Flash to create the structured business narrative
    final_analysis = ai_client.models.generate_content(
        model="gemini-1.5-flash",
        contents=synth_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.3  # Slightly higher temperature for fluid narrative synthesis
        )
    )
    
    # Return structured JSON containing the strategic narrative, query, and data preview
    return {
        "analysis": final_analysis.text,
        "sql_used": raw_sql,
        "data_preview": results[:10]
    }

In [38]:
# ------------------------------------------------------------------------------
# 5. Direct Execution Block (for Local Terminal Runs)
# ------------------------------------------------------------------------------

# Check if the script is being run directly (via `python app.py`) rather than imported as a module
if __name__ == "__main__":
    # Import uvicorn locally to spin up the ASGI server
    import uvicorn
    
    # Start the server listening on localhost at port 8000
    uvicorn.run(app, host="127.0.0.1", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop